## Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import math
import re
import pickle
from collections import Counter
from typing import Tuple, Dict, List, Optional
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Install packages and setup Colab environment
import sys
import subprocess

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run(['pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio', 'scikit-learn', 'pandas', 'matplotlib', 'seaborn', 'tqdm'], check=True)

print("✓ Environment ready!")

In [ ]:
class Config:
    """Configuration for training and inference."""
    
    # Model config
    VOCAB_SIZE = 30522
    D_MODEL = 256
    NUM_LAYERS = 4
    NUM_HEADS = 4
    D_FF = 1024
    MAX_SEQ_LENGTH = 256
    NUM_CLASSES = 2
    
    # Training config
    BATCH_SIZE = 32
    LEARNING_RATE = 2e-5
    NUM_EPOCHS = 3
    WARMUP_STEPS = 500
    
    # Data paths
    FAKE_CSV = 'Fake.csv'
    REAL_CSV = 'True.csv'
    
    # Device
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Config: D_MODEL={Config.D_MODEL}, Layers={Config.NUM_LAYERS}")
print(f"Device: {Config.DEVICE.upper()}")

## Configuration

# Fake News Detection using BERT Transformer
## GPU-Accelerated Training on Google Colab

**Estimated training time: ~15-30 minutes on Colab T4 GPU**

## Section 8: Create & Train Model

Run this cell to train the model on GPU (Colab) or CPU (local)

## Section 7: Data Loading & Preparation

In [ ]:
class FakeNewsTrainer:
    """Trainer class for fake news detection."""
    
    def __init__(self, model, train_loader, val_loader, test_loader, 
                 device='cpu', learning_rate=2e-5, num_epochs=3):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.device = device
        self.num_epochs = num_epochs
        
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
        
        total_steps = len(train_loader) * num_epochs
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=total_steps, eta_min=1e-6
        )
        
        self.train_losses = []
        self.val_losses = []
        self.val_accuracies = []
        
        self.model.to(device)
    
    def train_epoch(self):
        """Train for one epoch."""
        self.model.train()
        total_loss = 0.0
        
        progress_bar = tqdm(self.train_loader, desc="Training")
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(self.device)
            token_type_ids = batch['token_type_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            labels = batch['label'].to(self.device)
            
            logits, _ = self.model(input_ids, token_type_ids, attention_mask)
            loss = self.criterion(logits, labels)
            
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            self.scheduler.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': loss.item()})
        
        avg_loss = total_loss / len(self.train_loader)
        return avg_loss
    
    def evaluate(self, data_loader):
        """Evaluate on validation/test set."""
        self.model.eval()
        total_loss = 0.0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            progress_bar = tqdm(data_loader, desc="Evaluating")
            for batch in progress_bar:
                input_ids = batch['input_ids'].to(self.device)
                token_type_ids = batch['token_type_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['label'].to(self.device)
                
                logits, _ = self.model(input_ids, token_type_ids, attention_mask)
                loss = self.criterion(logits, labels)
                
                total_loss += loss.item()
                
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
        
        avg_loss = total_loss / len(data_loader)
        accuracy = accuracy_score(all_labels, all_preds)
        
        return avg_loss, accuracy, all_preds, all_labels
    
    def train(self):
        """Full training loop."""
        print(f"Starting training for {self.num_epochs} epochs on {self.device.upper()}...")
        
        for epoch in range(self.num_epochs):
            print(f"\n{'='*60}")
            print(f"Epoch {epoch+1}/{self.num_epochs}")
            print('='*60)
            
            train_loss = self.train_epoch()
            val_loss, val_acc, _, _ = self.evaluate(self.val_loader)
            
            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)
            self.val_accuracies.append(val_acc)
            
            print(f"\nTrain Loss: {train_loss:.4f}")
            print(f"Val Loss: {val_loss:.4f}")
            print(f"Val Accuracy: {val_acc:.4f}")
        
        # Test set evaluation
        print(f"\n{'='*60}")
        print("FINAL TEST RESULTS")
        print('='*60)
        test_loss, test_acc, test_preds, test_labels = self.evaluate(self.test_loader)
        
        print(f"\nTest Loss: {test_loss:.4f}")
        print(f"Test Accuracy: {test_acc:.4f}")
        print(f"Precision: {precision_score(test_labels, test_preds):.4f}")
        print(f"Recall: {recall_score(test_labels, test_preds):.4f}")
        print(f"F1-Score: {f1_score(test_labels, test_preds):.4f}")
        
        return {
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
            'val_accuracies': self.val_accuracies,
            'test_accuracy': test_acc,
            'test_preds': test_preds,
            'test_labels': test_labels
        }

print("✓ Trainer class defined")

## Section 6: Training Class

In [ ]:
class FakeNewsDataset(Dataset):
    """PyTorch Dataset for fake news detection."""
    
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        encoded = self.tokenizer.encode(
            text,
            max_length=self.max_length,
            padding=True,
            truncation=True
        )
        
        return {
            'input_ids': torch.tensor(encoded['input_ids'], dtype=torch.long),
            'token_type_ids': torch.tensor(encoded['token_type_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(encoded['attention_mask'], dtype=torch.float32),
            'label': torch.tensor(label, dtype=torch.long)
        }


def load_fake_news_data(fake_csv, real_csv, test_size=0.2, val_size=0.1):
    """Load fake news data from CSV files."""
    from sklearn.model_selection import train_test_split
    
    print("Loading data...")
    fake_df = pd.read_csv(fake_csv)
    real_df = pd.read_csv(real_csv)
    
    # Prepare texts and labels
    texts = []
    labels = []
    
    # Fake news
    for text in fake_df.get('text', fake_df.get('content', [])):
        if isinstance(text, str):
            texts.append(text)
            labels.append(1)  # Fake = 1
    
    # Real news
    for text in real_df.get('text', real_df.get('content', [])):
        if isinstance(text, str):
            texts.append(text)
            labels.append(0)  # Real = 0
    
    # Split data
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        texts, labels, test_size=test_size, random_state=42
    )
    
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        train_texts, train_labels, test_size=val_size, random_state=42
    )
    
    return train_texts, train_labels, val_texts, val_labels, test_texts, test_labels

print("✓ Dataset classes defined")

In [ ]:
# Check if running on Colab
import os

if IS_COLAB:
    print("Running on Google Colab!")
    print("\n📥 To load data, upload Fake.csv and True.csv:")
    print("   1. Click folder icon on left sidebar")
    print("   2. Click upload icon")
    print("   3. Select Fake.csv and True.csv from your computer")
    print("\n   Or download from Kaggle Fake and Real News Dataset")
    print("   https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset")

# Check if files exist
fake_path = 'Fake.csv'
real_path = 'True.csv'

if not os.path.exists(fake_path) or not os.path.exists(real_path):
    print(f"\n❌ Files not found:")
    print(f"   - {fake_path}: {os.path.exists(fake_path)}")
    print(f"   - {real_path}: {os.path.exists(real_path)}")
    print("\n⚠️  Please upload the CSV files to continue.")
else:
    print("\n✓ Loading fake news data...")
    train_texts, train_labels, val_texts, val_labels, test_texts, test_labels = \
        load_fake_news_data(fake_path, real_path)

    print(f"Dataset loaded:")
    print(f"  Train: {len(train_texts)} samples")
    print(f"  Val: {len(val_texts)} samples")
    print(f"  Test: {len(test_texts)} samples")

    # Initialize tokenizer
    print("\nInitializing tokenizer...")
    tokenizer = BERTTokenizer()
    tokenizer.build_vocab(train_texts, vocab_size=Config.VOCAB_SIZE)

    print(f"Tokenizer ready with vocab size: {len(tokenizer.vocab)}")

    # Create datasets
    print("\nCreating datasets...")
    train_dataset = FakeNewsDataset(train_texts, train_labels, tokenizer, Config.MAX_SEQ_LENGTH)
    val_dataset = FakeNewsDataset(val_texts, val_labels, tokenizer, Config.MAX_SEQ_LENGTH)
    test_dataset = FakeNewsDataset(test_texts, test_labels, tokenizer, Config.MAX_SEQ_LENGTH)

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)

    print(f"DataLoaders created:")
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches: {len(val_loader)}")
    print(f"  Test batches: {len(test_loader)}")
    print("\n✓ Ready for training!")

## Section 7: Data Loading & Preparation

## Section 5: Dataset & DataLoaders

In [ ]:
import re
from collections import Counter
import pickle

class BERTTokenizer:
    """Custom BERT tokenizer."""
    
    def __init__(self, vocab_size=30000):
        self.vocab_size = vocab_size
        self.vocab = {'[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3}
        self.reverse_vocab = {v: k for k, v in self.vocab.items()}
    
    def build_vocab(self, texts, vocab_size=30000):
        """Build vocabulary from texts."""
        word_freq = Counter()
        
        for text in texts:
            words = self._tokenize_text(text)
            word_freq.update(words)
        
        # Add most frequent words to vocab
        for word, _ in word_freq.most_common(vocab_size - len(self.vocab)):
            if word not in self.vocab:
                idx = len(self.vocab)
                self.vocab[word] = idx
        
        self.reverse_vocab = {v: k for k, v in self.vocab.items()}
    
    def _tokenize_text(self, text):
        """Tokenize text into words."""
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', '', text)
        words = text.split()
        return words
    
    def tokenize(self, text):
        """Convert text to tokens."""
        words = self._tokenize_text(text)
        tokens = []
        for word in words:
            tokens.append(word)
        return tokens
    
    def convert_tokens_to_ids(self, tokens):
        """Convert tokens to IDs."""
        ids = []
        for token in tokens:
            ids.append(self.vocab.get(token, self.vocab['[UNK]']))
        return ids
    
    def encode(self, text, max_length=512, padding=True, truncation=True):
        """Encode text to input IDs."""
        tokens = self.tokenize(text)
        token_ids = self.convert_tokens_to_ids(tokens)
        
        if truncation and len(token_ids) > max_length - 2:
            token_ids = token_ids[:max_length - 2]
        
        # Add [CLS] at start and [SEP] at end
        token_ids = [self.vocab['[CLS]']] + token_ids + [self.vocab['[SEP]']]
        
        if padding and len(token_ids) < max_length:
            token_ids = token_ids + [self.vocab['[PAD]']] * (max_length - len(token_ids))
        
        attention_mask = [1 if token != self.vocab['[PAD]'] else 0 for token in token_ids]
        token_type_ids = [0] * len(token_ids)
        
        return {
            'input_ids': token_ids,
            'attention_mask': attention_mask,
            'token_type_ids': token_type_ids
        }

print("✓ Tokenizer defined")

## Section 4: Tokenizer

In [ ]:
class PositionalEncoding(nn.Module):
    """Positional encoding for transformer."""
    
    def __init__(self, d_model: int, max_seq_length: int = 512):
        super().__init__()
        self.d_model = d_model
        self.max_seq_length = max_seq_length
        
        # Create positional encoding matrix
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1), :].to(x.device)


class MultiHeadAttention(nn.Module):
    """Multi-head self-attention mechanism."""
    
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.d_k)
    
    def forward(self, Q, K, V, mask=None):
        batch_size = Q.shape[0]
        
        # Linear projections
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # Attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = torch.nan_to_num(attention_weights, nan=0.0)
        attention_weights = self.dropout(attention_weights)
        
        # Apply attention to values
        output = torch.matmul(attention_weights, V)
        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, -1, self.d_model)
        
        return self.W_o(output), attention_weights


class FeedForward(nn.Module):
    """Feed-forward network."""
    
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.linear2(self.dropout(torch.relu(self.linear1(x))))


class TransformerEncoderLayer(nn.Module):
    """Single transformer encoder layer."""
    
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        
        self.mha = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        
        self.layernorm1 = nn.LayerNorm(d_model)
        self.layernorm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Multi-head attention with residual connection
        attn_output, _ = self.mha(x, x, x, mask)
        x = self.layernorm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual connection
        ffn_output = self.ffn(x)
        x = self.layernorm2(x + self.dropout(ffn_output))
        
        return x


class BERTTransformer(nn.Module):
    """BERT-like transformer model."""
    
    def __init__(self, vocab_size, d_model=256, num_layers=4, num_heads=4, 
                 d_ff=1024, max_seq_length=512, dropout=0.1, num_classes=2):
        super().__init__()
        
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.max_seq_length = max_seq_length
        
        # Embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        self.segment_embedding = nn.Embedding(2, d_model)
        self.embedding_dropout = nn.Dropout(dropout)
        self.embedding_ln = nn.LayerNorm(d_model)
        
        # Transformer encoder stack
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # Classification head
        self.pooler_dense = nn.Linear(d_model, d_model)
        self.pooler_activation = nn.Tanh()
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )
    
    def forward(self, input_ids, token_type_ids=None, attention_mask=None):
        # Embeddings
        seq_length = input_ids.size(1)
        token_embeds = self.token_embedding(input_ids)
        
        pos_embeds = self.positional_encoding(token_embeds[:, :seq_length, :])
        
        if token_type_ids is None:
            token_type_ids = torch.zeros_like(input_ids)
        
        segment_embeds = self.segment_embedding(token_type_ids)
        embeddings = pos_embeds + segment_embeds
        embeddings = self.embedding_dropout(embeddings)
        embeddings = self.embedding_ln(embeddings)
        
        # Prepare attention mask
        if attention_mask is not None:
            attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)
        
        # Pass through encoder layers
        hidden_states = embeddings
        for encoder_layer in self.encoder_layers:
            hidden_states = encoder_layer(hidden_states, attention_mask)
        
        # Pooling (use [CLS] token - first token)
        cls_output = hidden_states[:, 0, :]
        pooled_output = self.pooler_activation(self.pooler_dense(cls_output))
        
        # Classification
        logits = self.classifier(pooled_output)
        
        return logits, hidden_states

print("✓ BERT model defined")

## Section 3: BERT Model Implementation

Custom BERT transformer built from scratch with attention mechanisms.

In [ ]:
class Config:
    """Configuration for training and inference."""
    
    # Model config
    VOCAB_SIZE = 30522
    D_MODEL = 256  # Reduced for faster training
    NUM_LAYERS = 4  # Reduced for faster training
    NUM_HEADS = 4   # Must divide D_MODEL
    D_FF = 1024
    MAX_SEQ_LENGTH = 256  # Reduced for faster processing
    NUM_CLASSES = 2
    
    # Training config - Optimized for GPU
    BATCH_SIZE = 32  # Can be larger on GPU
    LEARNING_RATE = 2e-5
    NUM_EPOCHS = 3
    WARMUP_STEPS = 500
    
    # Data paths
    FAKE_CSV = 'Fake.csv'
    REAL_CSV = 'True.csv'
    
    # Output paths
    MODEL_SAVE_PATH = 'fake_news_bert_model.pt'
    TOKENIZER_PATH = 'bert_tokenizer.pkl'
    
    # Device
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Config: D_MODEL={Config.D_MODEL}, Layers={Config.NUM_LAYERS}, Batch Size={Config.BATCH_SIZE}")
print(f"Device: {Config.DEVICE.upper()}")

## Section 2: Configuration & Hyperparameters

Optimized for fast training on GPU while maintaining model quality.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from typing import Tuple, Dict, List, Optional
import time
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import math

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")